
# Qualidade de Dados — Interrupções de Carga do ONS

## Objetivo

Avaliar a qualidade dos dados utilizados no pipeline de Engenharia de Dados, verificando a confiabilidade dos registros desde a captura na Bronze até sua disponibilização na Gold.

A avaliação considera cinco dimensões:

1. Completude: presença de valores nulos ou vazios.
2. Consistência: conformidade com formatos e domínios esperados.
3. Unicidade: identificação de duplicações indevidas.
4. Acurácia e plausibilidade: coerência dos valores com o contexto elétrico.
5. Outliers: identificação de valores extremos que podem afetar as análises.

## Estratégia

A qualidade será avaliada em dois momentos:

- Bronze: diagnóstico dos dados originais, sem modificações.
- Silver: verificação dos dados após limpeza e aplicação das regras de negócio.

Os resultados serão utilizados para avaliar a eficácia das transformações e documentar eventuais limitações da fonte.


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce

CATALOG = "ons_energia"

df_bronze = spark.table(
    f"{CATALOG}.bronze.bronze_interrupcao_carga"
)

df_silver = spark.table(
    f"{CATALOG}.silver.silver_interrupcao_carga"
)

df_invalidos = spark.table(
    f"{CATALOG}.silver.silver_registros_invalidos"
)

df_fato = spark.table(
    f"{CATALOG}.gold.fato_interrupcao"
)

colunas_originais = [
    "cod_perturbacao",
    "din_interrupcaocarga",
    "id_subsistema",
    "nom_subsistema",
    "id_estado",
    "nom_agente",
    "val_cargainterrompida_mw",
    "val_tempomedio_minutos",
    "val_energianaosuprida_mwh",
    "flg_envolveuredebasica",
    "flg_envolveuredeoperacao"
]

print("Bronze:", df_bronze.count())
print("Silver:", df_silver.count())
print("Inválidos:", df_invalidos.count())
print("Gold:", df_fato.count())



## 1. Completude

A completude representa a proporção de registros que apresentam valores preenchidos em cada atributo.

Serão considerados ausentes:

- Valores nulos.
- Strings vazias.
- Strings contendo apenas espaços.

A análise será realizada individualmente para os 11 atributos originais, permitindo identificar quais campos apresentam maior proporção de informações ausentes.


In [0]:

total_bronze = df_bronze.count()

resultados = []

for coluna in colunas_originais:

    ausentes = df_bronze.filter(
        F.col(coluna).isNull() |
        (F.trim(F.col(coluna).cast("string")) == "")
    ).count()

    resultados.append((
        coluna,
        total_bronze,
        ausentes,
        round(
            100 * ausentes / total_bronze,
            4
        )
    ))

df_completude = spark.createDataFrame(
    resultados,
    [
        "atributo",
        "total_registros",
        "valores_ausentes",
        "percentual_ausencia"
    ]
)

display(
    df_completude.orderBy(
        F.desc("percentual_ausencia")
    )
)



## 2. Consistência

A consistência avalia se os valores respeitam os formatos e domínios definidos para cada atributo.

Serão verificadas:

- Estrutura dos códigos de perturbação.
- Datas válidas.
- Siglas dos estados.
- Identificação dos subsistemas.
- Indicadores de rede com valores S ou N.
- Conversibilidade dos atributos numéricos.

As regras de consistência serão comparadas com os tratamentos já implementados na camada Silver.


In [0]:

for coluna in [
    "id_subsistema",
    "nom_subsistema",
    "flg_envolveuredebasica",
    "flg_envolveuredeoperacao"
]:

    print(f"\nATRIBUTO: {coluna}")

    display(
        df_bronze
        .groupBy(coluna)
        .count()
        .orderBy(F.desc("count"))
    )


In [0]:

df_codigos = (
    df_bronze
    .withColumn(
        "formato_valido",
        F.col("cod_perturbacao").rlike(
            r"^[0-9]+/[0-9]{4}$"
        )
    )
    .groupBy("formato_valido")
    .count()
)

display(df_codigos)


In [0]:

colunas_erro = [
    "erro_identificador",
    "erro_data",
    "erro_subsistema",
    "erro_uf",
    "erro_carga",
    "erro_tempo",
    "erro_energia",
    "erro_rede_basica",
    "erro_rede_operacao"
]

df_erros = df_invalidos.agg(
    *[
        F.sum(
            F.when(F.col(c), 1).otherwise(0)
        ).alias(c)
        for c in colunas_erro
    ]
)

display(df_erros)



## 3. Unicidade

A análise de unicidade busca identificar registros repetidos indevidamente.

O código da perturbação não será tratado como identificador exclusivo de linha, pois uma mesma perturbação pode envolver diferentes agentes ou localidades.

A verificação será realizada considerando os 11 atributos originais.

Também será avaliada a unicidade da chave substituta da tabela fato, garantindo que cada registro válido possua um identificador exclusivo.


In [0]:

total = df_bronze.count()

distintos = (
    df_bronze
    .select(*colunas_originais)
    .distinct()
    .count()
)

duplicados = total - distintos

print("Registros originais:", total)
print("Registros distintos:", distintos)
print("Duplicatas:", duplicados)

print(
    "Percentual de duplicação:",
    round(100 * duplicados / total, 4),
    "%"
)

display(
    df_bronze
    .groupBy(*colunas_originais)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)


In [0]:

total_fato = df_fato.count()

chaves_distintas = (
    df_fato
    .select("sk_interrupcao")
    .distinct()
    .count()
)

assert total_fato == chaves_distintas

print("Registros da fato:", total_fato)
print("Chaves distintas:", chaves_distintas)
print("Unicidade aprovada.")



## 4. Acurácia e plausibilidade física

A acurácia representa a proximidade entre o valor registrado e o valor real.

Como não há uma fonte independente de referência para cada interrupção, não é possível comprovar integralmente a acurácia dos registros.

Será realizada uma avaliação de plausibilidade física, considerando que:

- Carga interrompida não deve ser negativa.
- Tempo médio de recomposição não deve ser negativo.
- Energia não suprida não deve ser negativa.

Valores extremos positivos não serão automaticamente considerados incorretos, pois grandes perturbações podem produzir impactos elevados.

A análise será complementada pela identificação estatística de outliers.


In [0]:

medidas = [
    "val_cargainterrompida_mw",
    "val_tempomedio_minutos",
    "val_energianaosuprida_mwh"
]

for coluna in medidas:

    print(f"\nMEDIDA: {coluna}")

    display(
        df_silver.agg(
            F.min(coluna).alias("minimo"),
            F.max(coluna).alias("maximo"),
            F.avg(coluna).alias("media"),
            F.expr(
                f"percentile_approx({coluna}, 0.5)"
            ).alias("mediana"),
            F.sum(
                F.when(
                    F.col(coluna) < 0, 1
                ).otherwise(0)
            ).alias("valores_negativos")
        )
    )



## 5. Identificação de Outliers

Será utilizado o método do intervalo interquartil (IQR) para identificar valores extremos nas três medidas quantitativas.

A identificação estatística não significa necessariamente erro de registro.

No contexto elétrico, interrupções de grande magnitude podem representar eventos reais e relevantes.

Por esse motivo, os outliers serão sinalizados e analisados, mas não removidos automaticamente.

A decisão preserva a capacidade do modelo de identificar perturbações críticas.


In [0]:

resultados_outliers = []

for coluna in medidas:

    q1, q3 = df_silver.approxQuantile(
        coluna,
        [0.25, 0.75],
        0.001
    )

    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    qtd_outliers = df_silver.filter(
        (F.col(coluna) < limite_inferior) |
        (F.col(coluna) > limite_superior)
    ).count()

    resultados_outliers.append((
        coluna,
        float(q1),
        float(q3),
        float(iqr),
        float(limite_inferior),
        float(limite_superior),
        int(qtd_outliers),
        round(
            100 * qtd_outliers / df_silver.count(),
            4
        )
    ))

df_outliers = spark.createDataFrame(
    resultados_outliers,
    [
        "atributo",
        "q1",
        "q3",
        "iqr",
        "limite_inferior",
        "limite_superior",
        "qtd_outliers",
        "percentual_outliers"
    ]
)

display(df_outliers)



## 6. Investigação da granularidade das medidas

A tabela fato possui granularidade de registro individual de interrupção, enquanto o código da perturbação pode se repetir.

Antes de agregar a energia não suprida, será investigado o comportamento dessa medida em perturbações com múltiplos registros.

A análise verificará:

- Quantidade de registros por perturbação.
- Quantidade de valores distintos de energia não suprida.
- Distribuição dos registros por agente e localidade.
- Possibilidade de repetição de medidas associadas à mesma perturbação.

Essa verificação busca evitar dupla contagem e assegurar que os indicadores finais respeitem a granularidade da fonte.


In [0]:

df_granularidade = (
    df_silver
    .groupBy("cod_perturbacao")
    .agg(
        F.count("*").alias("qtd_registros"),
        F.countDistinct(
            "val_energianaosuprida_mwh"
        ).alias("valores_distintos_ens"),
        F.countDistinct(
            "nom_agente"
        ).alias("qtd_agentes"),
        F.countDistinct(
            "id_estado"
        ).alias("qtd_estados"),
        F.sum(
            "val_energianaosuprida_mwh"
        ).alias("soma_ens_mwh")
    )
)

display(
    df_granularidade
    .orderBy(F.desc("qtd_registros"))
    .limit(20)
)


In [0]:

display(
    df_granularidade
    .filter(F.col("qtd_registros") > 1)
    .groupBy(
        "qtd_registros",
        "valores_distintos_ens"
    )
    .count()
    .orderBy(
        "qtd_registros",
        "valores_distintos_ens"
    )
)


In [0]:

perturbacao_exemplo = (
    df_granularidade
    .filter(F.col("qtd_registros") > 1)
    .orderBy(F.desc("qtd_registros"))
    .first()["cod_perturbacao"]
)

print("Perturbação analisada:", perturbacao_exemplo)

display(
    df_silver
    .filter(
        F.col("cod_perturbacao") ==
        perturbacao_exemplo
    )
    .select(
        "cod_perturbacao",
        "din_interrupcaocarga",
        "id_estado",
        "nom_agente",
        "val_cargainterrompida_mw",
        "val_tempomedio_minutos",
        "val_energianaosuprida_mwh"
    )
    .orderBy("din_interrupcaocarga")
)



## Persistência dos indicadores de qualidade

Os diagnósticos de qualidade serão armazenados em tabelas Delta no schema Gold.

Essa persistência permite consultar os resultados posteriormente, gerar evidências e comparar a qualidade dos dados entre execuções do pipeline.

As tabelas de qualidade não integram diretamente o esquema estrela analítico, mas constituem produtos auxiliares de monitoramento e governança.


In [0]:

tabelas_qualidade = {
    "qualidade_completude": df_completude,
    "qualidade_outliers": df_outliers,
    "qualidade_granularidade": df_granularidade
}

for nome, dataframe in tabelas_qualidade.items():

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"{CATALOG}.gold.{nome}"
        )
    )

    print(f"Tabela persistida: {nome}")


In [0]:

df_divergencia_ano = (
    df_silver
    .withColumn(
        "ano_codigo",
        F.regexp_extract(
            F.col("cod_perturbacao"),
            r"/(\d{4})$",
            1
        ).cast("int")
    )
    .withColumn(
        "ano_data",
        F.year("din_interrupcaocarga")
    )
    .filter(
        F.col("ano_codigo") != F.col("ano_data")
    )
)

print(
    "Registros com divergência:",
    df_divergencia_ano.count()
)

display(
    df_divergencia_ano.select(
        "cod_perturbacao",
        "din_interrupcaocarga",
        "ano_codigo",
        "ano_data"
    ).orderBy("din_interrupcaocarga")
)


In [0]:

display(
    df_silver.select(
        "cod_perturbacao",
        "din_interrupcaocarga",
        "id_estado",
        "nom_agente",
        "val_cargainterrompida_mw",
        "val_tempomedio_minutos",
        "val_energianaosuprida_mwh"
    )
    .orderBy(
        F.desc("val_tempomedio_minutos")
    )
    .limit(10)
)



## Conclusões da avaliação de qualidade

A avaliação dos 8.326 registros originais demonstrou completude de 100% nos 11 atributos da fonte.

Foram identificados 16 registros integralmente duplicados e um registro com valores fisicamente inconsistentes de tempo médio de recomposição e energia não suprida.

Após o tratamento, a camada Silver disponibilizou 8.309 registros válidos, integralmente reconciliados com a camada Bronze.

A análise estatística identificou 743 outliers de carga interrompida, 885 de tempo médio e 1.116 de energia não suprida.

Os valores extremos positivos foram preservados, pois podem representar perturbações reais de grande magnitude.

A investigação da granularidade demonstrou que uma perturbação pode envolver múltiplos agentes, localidades e registros com diferentes valores de energia não suprida.

Por esse motivo, a quantidade de perturbações será calculada por identificadores distintos, enquanto as medidas serão analisadas respeitando a granularidade dos registros.

A acurácia dos valores individuais não pôde ser integralmente comprovada por ausência de uma fonte independente de comparação.

As limitações identificadas serão consideradas na interpretação dos indicadores e nas conclusões finais do MVP.



### Verificações complementares

Foram identificados 201 registros nos quais o ano contido no código da perturbação difere do ano da data registrada. A divergência foi preservada para fins de rastreabilidade, sem correção automática, pois não foi possível determinar sua causa a partir dos dados disponíveis.

As análises temporais utilizarão a data da interrupção, e não o ano incorporado ao identificador.

Também foram observados tempos médios de recomposição extremamente elevados. O maior valor corresponde a aproximadamente 47,3 dias. Esses registros foram mantidos, pois a magnitude estatística, isoladamente, não comprova erro.

Nas análises de duração, serão apresentadas média e mediana para explicitar a influência dos valores extremos.
